<div style="background-color:#0B3C5D; padding:25px; border-radius:10px;">

<table style="width:100%; border-collapse:collapse;">
<tr>
<td style="width:20%; text-align:left;">
    <img src="https://surfdrive.surf.nl/s/nA3dfeS4PwZ6H43/download?path=%2F&files=/ihe-delft-institute_unesco_fc.png" height="85">
</td>

<td style="width:60%; text-align:center;">
    <h1 style="color:white; margin:0;">
        MOOC: <b>WaPOR for Global Challenges</b>
    </h1>
</td>

<td style="width:20%; text-align:right;">
    <img src="https://surfdrive.surf.nl/s/nA3dfeS4PwZ6H43/download?path=%2F&files=/waporlogo.png" height="85">
</td>
</tr>
</table>

<div style="text-align:center; margin-top:15px;">
<h3 style="color:#D9E6F2; margin-bottom:5px;">
<i>Module Two</i> – <b>Data Access with Python and GEE</b>
</h3>

<p style="color:white; margin-top:5px;">
<b>Notebook:</b> Download WaPOR using GEE &nbsp; | &nbsp;
<b>Estimated time:</b> 2 hours &nbsp; | &nbsp;
<b>Instructor:</b> Dr. Solomon Seyoum
</p>
</div>

</div>

With this notebook, we will explore how to access <font  color='#d39aed'> Level 1 </font> and <font color='#d39aed'> Level 2 </font> WaPOR data using Google Earth Engine in Python.




<div style="border-left:6px solid #1D70B8; background-color:#0B3C5D; padding:15px; border-radius:6px">

### 🎯 Learning Objectives

When you complete this notebook, you will be able to:

- Explain the different ways how WaPOR data can be accessed  
- Understand the code in the notebook
- Access WaPOR data and download the data for the example area of interest  
- Modify the code to access and download the data for your need    

</div>

<div style="background-color:#0B3C5D; padding:15px; border-radius:6px">

### 📝 Prerequisites

✔ [Introduction to WaPOR](https://ocw.un-ihe.org/course/view.php?id=263)  
✔ Geospatial data concepts  
✔ [Python basics for geospatial  data analysis](https://ocw.un-ihe.org/course/view.php?id=272)   
✔ Gmail account   
✔ GEE basics    

</div>

---

###  🔀 Workflow Overview
🗺

| Step | Task | Purpose |
|------|------|---------|
|   | Environment Setup | importing required packages
| 2 | Input Data Preparation | Set up the input required to run the download script |
| 3 | Running the download script | Download the data |
| 4 | Check| Check if the required data is downloaded based on the specification |
| 5 | Optional - Visualize | Visualize the downloaded data |

---

<div style="background-color:#0B3C5D; padding:18px; border-radius:6px">

## 📖 WaPOR data access

### WaPOR data can be accessed in several ways:

- <h4 style="margin-bottom:1px;">FAO WaPOR Portal</h4>
	<p style="margin-top:0px;">
	The simplest way to access WaPOR data is through the FAO portal for monitoring agricultural water productivity using open-access, remotely sensed data. The process is demonstrated in the <a href=(https://ocw.un-ihe.org/course/view.php?id=263)" target="_blank">Introduction to WaPOR</a> OpenCourseWare. This portal is user-friendly and ideal for quickly accessing small datasets. For larger volumes of data, other methods are more efficient.
	</p>

- <h4 style="margin-bottom:1px;">Google Earth Engine (GEE)</h4>
    <p style="margin-top:0px;">
	WaPOR Version 3 data is available in GEE, enabling faster cloud-based access, processing, and analysis. Instructions for using GEE to access WaPOR data are covered in the next notebook.
	</p>

- <h4 style="margin-bottom:1px;">Google Cloud Storage</h4>
    <p style="margin-top:0px;">
	WaPOR data is also hosted on Google Cloud Storage and can be accessed programmatically to download data for specific areas of interest or directly per file. This <a href=(https://bitbucket.org/cioapps/wapordl/src/main/)" target="_blank">python package</a> was developed to download WaPOR v3 data.
	</p>

- <h4 style="margin-bottom:1px;">QGIS Plugins</h4>
    <p style="margin-top:0px;">
	Users can access WaPOR data through QGIS using plugins such as <a href="https://plugins.qgis.org/plugins/wap_plugin/" target="_blank">WaPlugin</a> and <a href="https://github.com/bvissers/FAO-Downloader" target="_blank">FAO downloader</a>.
	</p>

- <h4 style="margin-bottom:1px;">WaPOR v3 API</h4>
    <p style="margin-top:0px;">
	The script used in the previous notebook is built with the WaPOR Version 3 API to download and preprocess data for a selected area of interest.
	</p>

</div>


This notebook is to explore how to access WaPOR data using GEE. WaPOR level 1 and 2 data can be directly accessed in GEE as they are in Earth Engine Data Catalogue. Level 3 are not part of the Earth Engine Data Catalogue and; therefore, to access them, one first needs to upload the level 3 data to their google cloud storage. We will see how this can be done in this notebook.

 <div style="border-left:6px solid #1D70B8; background-color:#0B3C5D; padding:15px; border-radius:6px">

### The steps to access WaPOR data from GEE are:

When you complete this notebook, you will be able to:

- Install and import required libraries
- Authenticate and Initialize GEE and Google Cloud resource:
- Define study area
- Access the data
- Analysis and visualization ( this part will be covered in another notebook)  

</div>


## ⚙️ Environment Setup
<p style="margin-top:1px;">
  Some Python packages are required that may not be included in a basic Python installation. It is also good practice to use an organized folder structure, with your code in one folder and your data in another. The data folder can contain different formats of input and output files. <br><br>
  In the next cells, we will create the necessary folders, upload the download script, and install the required Python packages. <br><br>
  </p>


### Install require packages

In [1]:
# Install required package
%%capture
!pip install geemap google-cloud-storage earthengine-api rasterio numpy requests

### Import require packages

In [2]:
# import packages
import ee
import geemap
import re
from google.cloud import storage
import pandas as pd  # data manipulation and analysis
from pandas.tseries.api import guess_datetime_format
import numpy as npm
import json # for handling JSON data (serialization and deserialization)
import time  #  timing operations and delays

import matplotlib.pyplot as plt  # data visualization
import folium  #  interactive maps
from folium import Figure
import branca.colormap as cm  #  color mapping in folium

### Authenticate earth engine

You need to have your Google Cloud project ID and your username to authenticate ee.

In [3]:
project_id = 'ee-mmul'#'your GC project ID
user = 'mmul'
ee.Authenticate()  #
ee.Initialize(project=project_id)

### Define some parameters to be used later

In [57]:
asset_folder = f"users/{user}/WaPOR4Global" ##*your GEE Legacy Assets user name*##
rio_assets_id = f'projects/ee-mmul/assets/Erbil'

AOI_name = "Erbil"
vars_to_download = ['L2-AETI-D', 'L1-RET-D']

# Define date range
start_date = "2024-06-01"
end_date = "2024-10-31"

Note on WaPOR data access: the data is not yet findable in earth engine data catalogue through its search function, but it can be accessed using the datasets "collection identifiers". These identifiers are:

- projects/UNFAO/wapor/v3/L1-AETI-D
- projects/UNFAO/wapor/v3/L1-E-D
- projects/UNFAO/wapor/v3/L1-I-D
- projects/UNFAO/wapor/v3/L1-T-D
- projects/UNFAO/wapor/v3/L1-NPP-D
- projects/UNFAO/wapor/v3/L1-RET-D
- projects/UNFAO/wapor/v3/L1-RET-E **
- projects/UNFAO/wapor/v3/L2-AETI-D
- projects/UNFAO/wapor/v3/L2-E-D
- projects/UNFAO/wapor/v3/L2-I-D
- projects/UNFAO/wapor/v3/L2-T-D
- projects/UNFAO/wapor/v3/L2-NPP-D

be aware that pprecipitation and rootzone mositure data are not avilable even for Level 1 and level 2 data.


### Get the image collections for the required variables and the date specified

In [58]:

var_paths_dict = {v: f"projects/UNFAO/wapor/v3/{v}" for v in vars_to_download}
def get_image_wapor(v):
    return (
      ee.ImageCollection(v)
      .filterDate(start_date, end_date)
      .map(lambda img: img.toInt()
          .copyProperties(img, img.propertyNames())
          .set('date', img.date().format('YYYY-MM')))
    )

var_collections_dict = {k: get_image_wapor(v) for k, v in var_paths_dict.items()}

In [59]:
var_collections_dict['L2-AETI-D']

### Read the area of interest

In [46]:
area_of_interest = ee.FeatureCollection(rio_assets_id)

### get information about the image collection such as band names projection and scale

In [60]:
def get_image_coll_info(img_cols):
  first_image = ee.Image(img_cols.first())
  band_names = first_image.bandNames()
  projection = first_image.projection()
  scale = first_image.projection().nominalScale()
  return  projection, scale, band_names

for v in var_collections_dict.keys():
  projection, scale, band_names = get_image_coll_info(var_collections_dict[v])
  print(f"Band_names: {band_names.getInfo()} projection: {projection.getInfo()}, scaLe: {scale.getInfo()}")

Band_names: ['L2-AETI-D'] projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.0009765625, 0, -180, 0, -0.0009765625, 60]}, scaLe: 108.71044022780622
Band_names: ['L1-RET-D'] projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.3125, 0, -180.15625, 0, -0.25, 90.125]}, scaLe: 31114.74357930271


### Clip the image collection using the area of interest

In [61]:
# Function to clip each image
var_collections_clipped_dict = {k: ee.ImageCollection(v).filterBounds(area_of_interest).map(lambda img: img.clip(area_of_interest)) for k, v in var_collections_dict.items()}
# var_collections_clipped_dict

### visualize a sample image

In [62]:
#  min and max
min_val, max_val = 0, 100

# Set visualization parameters
abst_vis_params = {
    "min": min_val,
    "max":max_val,
    "palette": [
       '#d7191c',
       '#fdae61',
       '#ffffbf',
       '#a6d96a',
       '#1a9641'
  ]
}
var_to_map = [x for x in vars_to_download if "AETI" in x]
Map = geemap.Map(height="600px", width="800px")
if len(var_to_map) >0:
  img = var_collections_clipped_dict[var_to_map[0]].first()
  Map.centerObject(area_of_interest, 7)
  Map.addLayer(img, abst_vis_params, "WaPOR AETI")
  Map.addLayerControl()

Map

Map(center=[36.34423670990206, 44.23309765651537], controls=(WidgetControl(options=['position', 'transparent_b…

In [50]:
#  min and max
min_val, max_val = 0, 100

# Set visualization parameters
abst_vis_params = {
    "min": min_val,
    "max":max_val,
    "palette": ["blue", "green", "yellow", "orange", "red"],
}


var_to_map = [x for x in vars_to_download if "RET" in x]
Map = geemap.Map(height="600px", width="800px")
if len(var_to_map) >0:
  img = var_collections_clipped_dict[var_to_map[0]].first()
  Map.centerObject(area_of_interest, 7)
  Map.addLayer(img, abst_vis_params, "WaPOR RET")
  Map.addLayerControl()

Map

Map(center=[36.344236709902056, 44.23309765651536], controls=(WidgetControl(options=['position', 'transparent_…

In [51]:
# As you can see the spatial resolution of the AETI and RET data are different.
# AETI level 2 has 100m spatial resolution where as RET has about 30 km
# the resolution should be the same before carring out any analysis.


def resample_to_target(input_data, target_img, method='bilinear'):
    """
    Resample an ee.Image or ee.ImageCollection to match a target image.

    Args:
        input_data (ee.Image or ee.ImageCollection): Input image(s) to resample.
        target_img (ee.Image): Target image whose projection, scale, and CRS to match.
        method (str): Resampling method: 'bilinear' (default) or 'nearest'.

    Returns:
        ee.Image or ee.ImageCollection: Resampled image(s)
    """

    def resample_image(img):
        return img.resample(method).reproject(
            crs=target_img.projection(),
            scale=target_img.projection().nominalScale()
        )

    # Handle single image
    if isinstance(input_data, ee.image.Image):
        return resample_image(input_data)

    # Handle image collection
    elif isinstance(input_data, ee.imagecollection.ImageCollection):
        return input_data.map(resample_image)

    else:
        raise TypeError("input_data must be ee.Image or ee.ImageCollection")


var_to_map = [x for x in vars_to_download if "AETI" in x]
target_img = var_collections_clipped_dict[var_to_map[0]].first()
ic = var_collections_clipped_dict['L1-RET-D']

resampled_ic = resample_to_target(ic, target_img, method='bilinear')

### Precipitation (PCP) data from WaPOR is not available in GEE but CHRIPS is available.
If PCP data is needed, it can be downloaded and resample to match the temporal and spatial resolution of the WaPOR data.


Dwonload CHIRPS daily data

In [17]:
# CHIRPS daily data

CHIRPS_E = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(start_date, end_date)
    .map(lambda img: img.toInt()
         .copyProperties(img, img.propertyNames())
         .set('date', img.date().format('YYYY-MM')))
)

In [52]:
# get the list of dates from WaPOR data for aggregation the PCP data

# Sort the ImageCollection and convert to a list
AETI_img = var_collections_clipped_dict['L2-AETI-D']
aeti_list = AETI_img.sort('system:time_start').toList(AETI_img.size())


def extract_dates(img):
    img = ee.Image(img)

    start = ee.Date(img.get('system:time_start'))
    end   = ee.Date(img.get('system:time_end'))

    nDays = end.difference(start, 'day')

    return ee.Dictionary({
        'startDate': start.format('YYYY-MM-dd'),
        'endDate': end.format('YYYY-MM-dd'),
        'nDays': nDays
    })

dateRanges = aeti_list.map(extract_dates)

print('WaPOR dekadal date ranges:')
print(dateRanges.getInfo())

WaPOR dekadal date ranges:
[{'endDate': '2024-11-10', 'nDays': 9, 'startDate': '2024-11-01'}, {'endDate': '2024-11-20', 'nDays': 9, 'startDate': '2024-11-11'}, {'endDate': '2024-11-30', 'nDays': 9, 'startDate': '2024-11-21'}, {'endDate': '2024-12-10', 'nDays': 9, 'startDate': '2024-12-01'}, {'endDate': '2024-12-20', 'nDays': 9, 'startDate': '2024-12-11'}, {'endDate': '2024-12-31', 'nDays': 10, 'startDate': '2024-12-21'}, {'endDate': '2025-01-10', 'nDays': 9, 'startDate': '2025-01-01'}, {'endDate': '2025-01-20', 'nDays': 9, 'startDate': '2025-01-11'}, {'endDate': '2025-01-31', 'nDays': 10, 'startDate': '2025-01-21'}, {'endDate': '2025-02-10', 'nDays': 9, 'startDate': '2025-02-01'}, {'endDate': '2025-02-20', 'nDays': 9, 'startDate': '2025-02-11'}, {'endDate': '2025-02-28', 'nDays': 7, 'startDate': '2025-02-21'}, {'endDate': '2025-03-10', 'nDays': 9, 'startDate': '2025-03-01'}, {'endDate': '2025-03-20', 'nDays': 9, 'startDate': '2025-03-11'}, {'endDate': '2025-03-31', 'nDays': 10, 'startD

In [63]:
# Function to aggregate daily images to the temporal resolution of waPOR AETI

def aggregate_collection(date_dicts, collection, agg_method='sum'):
    """
    Aggregate an ImageCollection over multiple date ranges.

    Args:
        date_dicts (ee.List or list of dict): each dict must have 'startDate', 'endDate', optionally 'nDays'
        collection (ee.ImageCollection): input collection
        agg_method (str): 'sum', 'mean', 'median', 'max', 'min'

    Returns:
        ee.ImageCollection
    """

    date_dicts = ee.List(date_dicts)

    def aggregate_one(d):
        d = ee.Dictionary(d)

        start = ee.Date(d.get('startDate'))
        end   = ee.Date(d.get('endDate')).advance(1, 'day')

        filtered = collection.filterDate(start, end)

        # Choose reducer
        aggregated = ee.Image(
            ee.Algorithms.If(agg_method == 'sum', filtered.sum(),
            ee.Algorithms.If(agg_method == 'mean', filtered.mean(),
            ee.Algorithms.If(agg_method == 'median', filtered.median(),
            ee.Algorithms.If(agg_method == 'max', filtered.max(),
            ee.Algorithms.If(agg_method == 'min', filtered.min())))))
        )

        return (aggregated
                .set('system:time_start', start.millis())
                .set('start_date', d.get('startDate'))
                .set('end_date', d.get('endDate'))
                .set('n_days', d.get('nDays')))

    return ee.ImageCollection(date_dicts.map(aggregate_one))

agg_pcp = aggregate_collection(dateRanges, CHIRPS_E, agg_method='sum')


# Optional: print info
print('Aggregated dekadal ImageCollection:')
print(agg_pcp.getInfo())

Aggregated dekadal ImageCollection:
{'type': 'ImageCollection', 'bands': [], 'features': [{'type': 'Image', 'bands': [{'id': 'precipitation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -9.223372036854776e+18, 'max': 9.223372036854776e+18}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 1730419200000, 'end_date': '2024-11-10', 'n_days': 9, 'start_date': '2024-11-01', 'system:index': '0'}}, {'type': 'Image', 'bands': [{'id': 'precipitation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -9.223372036854776e+18, 'max': 9.223372036854776e+18}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 1731283200000, 'end_date': '2024-11-20', 'n_days': 9, 'start_date': '2024-11-11', 'system:index': '1'}}, {'type': 'Image', 'bands': [{'id': 'precipitation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -9.223372036854776e+18, 'max': 9.223372036854776e+18}, 'cr

In [54]:
## resample the PCP
var_to_map = [x for x in vars_to_download if "AETI" in x]
target_img = var_collections_clipped_dict[var_to_map[0]].first()
ic = agg_pcp

resampled_pcp = resample_to_target(ic, target_img, method='bilinear')

Now since all the image are in the same temporal and spatial resolution, you can do the analysis.

---

To assess where irrigation is taking place during the dry season (June-October), we utilise the water deficit index (see FAO, 2018).

\Wdf=PCPs/AETIs
for Wdf > 0.9, no irrigation [0]
for WDf <= 0.9, irrigation [1]


In [64]:
#  min and max
min_val, max_val = 0, 500

# Set visualization parameters
abst_vis_params = {
    "min": min_val,
    "max":max_val,
    "palette": [
       '#d7191c',
       '#fdae61',
       '#ffffbf',
       '#a6d96a',
       '#1a9641'
  ]}

aeti_collection = var_collections_clipped_dict['L2-AETI-D']

# Filter the collection by the defined start and end dates
aeti_filtered = aeti_collection.filterDate(start_date, end_date)

# Sum the images in the filtered collection
summed_aeti = aeti_filtered.sum().clip(area_of_interest)

print(f"Summed AETI map (first band): {summed_aeti.bandNames().getInfo()}")

# Visualize the summed AETI map
Map_sum = geemap.Map(height="600px", width="800px")
Map_sum.centerObject(area_of_interest, 7)
Map_sum.addLayer(summed_aeti, abst_vis_params, "Summed WaPOR AETI")
# Add the colorbar (legend)
Map_sum.add_colorbar(abst_vis_params, label="Summed AETI Value", orientation="horizontal", layer_name="Summed WaPOR AETI")
Map_sum.addLayerControl()
Map_sum

Summed AETI map (first band): ['L2-AETI-D']


Map(center=[36.34423670990206, 44.23309765651537], controls=(WidgetControl(options=['position', 'transparent_b…

# 📊 Exercise
Please prepare shapefiles of your area of interest to download raster and point data. Upload the shapefiles and download raster data and point data for your area of interest.

